# Step 1 Hardware Profiling — UnifoLM-VLA-0 with NVIDIA Nsight Systems (G1_Clean_Table)

Runs **Nsight Systems (`nsys`) profiling** for the current focus task (**wipe/clean table**), capturing a hardware-level timeline of CUDA kernels, memory transfers, and NVTX ranges.

The generated `nsys_runner.py` uses the V100-optimized `UnifoLMVLAWrapper` (FP16, `torch.compile`, pinned async H2D). Post-run, compare `cudaLaunchKernel` and `cudaStreamSynchronize` counts against the pre-optimization baseline.

This notebook does **not** replace your PyTorch-profiler workflow in `step1_unifolm_profiling.ipynb`. It adds a second profiling path with source-labeled, timestamped artifacts.

Artifacts from this notebook use:
- `profiler_source = "nsight_systems"`
- `run_tag = nsight_systems_YYYYMMDD_HHMMSS`


In [1]:
from pathlib import Path
import os
import sys
import shutil

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    RESEARCH_DIR = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "src" / "step1_profile_unifolm_vla0.py").is_file():
    RESEARCH_DIR = NOTEBOOK_DIR
else:
    RESEARCH_DIR = NOTEBOOK_DIR / "research_summer_2026" / "research"
    if not RESEARCH_DIR.is_dir():
        RESEARCH_DIR = NOTEBOOK_DIR / "research"

RESEARCH_DIR = RESEARCH_DIR.resolve()
assert (RESEARCH_DIR / "src").is_dir(), f"src package not found under: {RESEARCH_DIR}"

os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results base  : {RESEARCH_DIR / 'results' / 'step1_profiling_unifolm_vla0'}")
print(f"Each run saves to: <results_base>/<profiler_source>_YYYYMMDD_HHMMSS/")
print(f"nsys binary   : {shutil.which('nsys')}")


Research root : /home/aihimekpen/research_summer_2026/research
Results go to : /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0
nsys binary   : /usr/local/cuda/bin/nsys


In [2]:
import json
import subprocess
from datetime import datetime
from textwrap import dedent

from src.step1_profile_unifolm_vla0 import RESULTS_BASE_DIR, make_run_results_dir

assert shutil.which("nsys"), (
    "Nsight Systems CLI (nsys) not found. Install Nsight Systems and ensure `nsys` is in PATH."
)

# Task focus for now. Update these later when moving beyond clean-table.
TASK_LABEL = "G1_Clean_Table"
INSTRUCTION = "Clean the table by moving all clutter items into the bin."
MODEL_ID = "unitreerobotics/UnifoLM-VLM-Base"
NSYS_WARMUP_STEPS = 1
NSYS_PROFILE_STEPS = 3

PROFILER_SOURCE = "nsight_systems"
RUN_TAG = f"{PROFILER_SOURCE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

NSYS_DIR = make_run_results_dir(PROFILER_SOURCE, RUN_TAG)
NSYS_OUTPUT_PREFIX = NSYS_DIR / "unifolm_vla0_nsys"
METADATA_JSON = NSYS_DIR / "nsight_systems_report.json"
RUNNER_PATH = NSYS_DIR / "nsys_runner.py"

print(f"TASK_LABEL      : {TASK_LABEL}")
print(f"PROFILER_SOURCE : {PROFILER_SOURCE}")
print(f"RUN_TAG         : {RUN_TAG}")
print(f"Run folder      : {NSYS_DIR}")
print(f"NSYS_OUTPUT     : {NSYS_OUTPUT_PREFIX}")


/home/aihimekpen/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


TASK_LABEL      : G1_Clean_Table
PROFILER_SOURCE : nsight_systems
RUN_TAG         : nsight_systems_20260617_115140
NSYS_OUTPUT     : /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/nsight_systems_runs/unifolm_vla0_nsys_nsight_systems_20260617_115140


In [4]:
runner_code = dedent(
    f"""
import os
import sys
from pathlib import Path

RESEARCH_DIR = Path({str(RESEARCH_DIR)!r})
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))
os.chdir(RESEARCH_DIR)

import torch
from src.step1_profile_unifolm_vla0 import MockG1CleanTableEnv, UnifoLMVLAWrapper

TASK_LABEL = {TASK_LABEL!r}
INSTRUCTION = {INSTRUCTION!r}
MODEL_ID = {MODEL_ID!r}
WARMUP_STEPS = {NSYS_WARMUP_STEPS}
PROFILE_STEPS = {NSYS_PROFILE_STEPS}

env = MockG1CleanTableEnv(image_size=(224, 224))
model = UnifoLMVLAWrapper(
    model_id=MODEL_ID,
    use_int4=False,
    action_dim=29,
    allow_mock_fallback=False,
)

obs = env.reset()
for _ in range(WARMUP_STEPS):
    torch.cuda.nvtx.range_push('vla_action_generation_warmup')
    action, _, _ = model.infer(obs, INSTRUCTION)
    torch.cuda.nvtx.range_pop()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    obs, _, done, _ = env.step(action)
    if done:
        obs = env.reset()

for _ in range(PROFILE_STEPS):
    torch.cuda.nvtx.range_push('vla_action_generation_profiled')
    action, _, _ = model.infer(obs, INSTRUCTION)
    torch.cuda.nvtx.range_pop()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    obs, _, done, _ = env.step(action)
    if done:
        obs = env.reset()

print('NSYS target run complete')
"""
)
RUNNER_PATH.write_text(runner_code, encoding="utf-8")

nsys_env = os.environ.copy()
nsys_env["PYTHONPATH"] = str(RESEARCH_DIR) + os.pathsep + nsys_env.get("PYTHONPATH", "")

cmd = [
    "nsys",
    "profile",
    "--trace=cuda,nvtx,osrt",
    "--sample=none",
    "--cpuctxsw=none",
    "--force-overwrite=true",
    "--output",
    str(NSYS_OUTPUT_PREFIX),
    "python3",
    str(RUNNER_PATH),
]

print("Running:", " ".join(cmd))
res = subprocess.run(cmd, cwd=RESEARCH_DIR, env=nsys_env, capture_output=True, text=True)
print("Exit code:", res.returncode)
print("--- stdout (tail) ---")
print("\n".join(res.stdout.splitlines()[-30:]))
print("--- stderr (tail) ---")
print("\n".join(res.stderr.splitlines()[-30:]))
if res.returncode != 0:
    raise RuntimeError("nsys profiling failed; inspect stderr above.")


Running: nsys profile --trace=cuda,nvtx,osrt --sample=none --cpuctxsw=none --force-overwrite=true --output /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/nsight_systems_runs/unifolm_vla0_nsys_nsight_systems_20260617_115140 python3 /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/nsight_systems_runs/nsys_runner_nsight_systems_20260617_115140.py
Exit code: 0
--- stdout (tail) ---
[1/1] [===============67%          ] unifolm_vla0_nsys_nsight_systems_20260617_115140.nsys-rep
[1/1] [================68%         ] unifolm_vla0_nsys_nsight_systems_20260617_115140.nsys-rep
[1/1] [================69%         ] unifolm_vla0_nsys_nsight_systems_20260617_115140.nsys-rep
[1/1] [================70%         ] unifolm_vla0_nsys_nsight_systems_20260617_115140.nsys-rep
[1/1] [================71%         ] unifolm_vla0_nsys_nsight_systems_20260617_115140.nsys-rep
[1/1] [=================72%        ] unifolm_vla0_nsys_nsight_systems_20

In [5]:
nsys_rep_path = NSYS_OUTPUT_PREFIX.with_suffix(".nsys-rep")
sqlite_path = NSYS_OUTPUT_PREFIX.with_suffix(".sqlite")
stats_txt_path = NSYS_DIR / f"nsight_systems_stats_{RUN_TAG}.txt"

stats_cmd = [
    "nsys",
    "stats",
    "--report",
    "cuda_gpu_trace,cuda_api_sum,gpu_kern_sum",
    str(nsys_rep_path),
]
stats_res = subprocess.run(stats_cmd, cwd=RESEARCH_DIR, capture_output=True, text=True)
stats_txt_path.write_text(stats_res.stdout + "\n\n--- STDERR ---\n" + stats_res.stderr, encoding="utf-8")

metadata = {
    "profiler_source": PROFILER_SOURCE,
    "run_tag": RUN_TAG,
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "task": TASK_LABEL,
    "instruction": INSTRUCTION,
    "model_id": MODEL_ID,
    "results_dir": str(NSYS_DIR),
    "nsys_rep": str(nsys_rep_path),
    "nsys_sqlite": str(sqlite_path),
    "nsys_stats_txt": str(stats_txt_path),
}
METADATA_JSON.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Profiler source:", PROFILER_SOURCE)
print("Run folder     :", NSYS_DIR)
print("NSYS rep exists:", nsys_rep_path.exists(), nsys_rep_path)
print("SQLite exists  :", sqlite_path.exists(), sqlite_path)
print("Stats txt      :", stats_txt_path)
print("Report JSON    :", METADATA_JSON)


Profiler source: nsight_systems
Run tag        : nsight_systems_20260617_115140
NSYS rep exists: True /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/nsight_systems_runs/unifolm_vla0_nsys_nsight_systems_20260617_115140.nsys-rep
SQLite exists  : True /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/nsight_systems_runs/unifolm_vla0_nsys_nsight_systems_20260617_115140.sqlite
Stats txt      : /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/nsight_systems_runs/nsight_systems_stats_nsight_systems_20260617_115140.txt
Report JSON    : /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/nsight_systems_runs/nsight_systems_report_nsight_systems_20260617_115140.json
